# 05 - First analysis (production, quality, maintenance)

With the data already cleaned in `datasets/processed/`, I now answer
the project's business questions with charts: where the plant is
losing the most efficiency, what the main stoppage causes are, how
quality looks, and how reliable the machines are.

Every chart is saved to `reports/` as a `.png`, so it can be reused
later in a final report or the dashboard.


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROCESSED = "../../datasets/processed"
REPORTS = "../../reports"
os.makedirs(REPORTS, exist_ok=True)

MAIN_COLOR = "#4C72B0"

production = pd.read_csv(f"{PROCESSED}/fact_production_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])
downtime = pd.read_csv(f"{PROCESSED}/fact_downtime_processed.csv", encoding="utf-8-sig", parse_dates=["Date"])

cap_variable = pd.read_csv(f"{PROCESSED}/fact_cap_inspection_variable_cq_processed.csv", encoding="utf-8-sig")
cap_attribute = pd.read_csv(f"{PROCESSED}/fact_cap_attribute_inspection_cq_processed.csv", encoding="utf-8-sig")
cap_disposition = pd.read_csv(f"{PROCESSED}/fact_cap_disposition_lot_cq_processed.csv", encoding="utf-8-sig")
bottle_attribute = pd.read_csv(f"{PROCESSED}/fact_bottle_attribute_inspection_cq_processed.csv", encoding="utf-8-sig")
bottle_disposition = pd.read_csv(f"{PROCESSED}/fact_bottle_disposition_lot_cq_processed.csv", encoding="utf-8-sig")
ink_attribute = pd.read_csv(f"{PROCESSED}/fact_ink_attribute_inspection_cq_processed.csv", encoding="utf-8-sig")
ink_disposition = pd.read_csv(f"{PROCESSED}/fact_ink_disposition_lot_cq_processed.csv", encoding="utf-8-sig")

print("production:", production.shape)
print("downtime: ", downtime.shape)


## 1. Production

### 1.1 OEE by process

OEE = Availability x Performance x Quality. Comparing the four
processes side by side immediately shows where the plant is losing the
most effective capacity.


In [ ]:
oee_by_process = production.groupby("Process")[["Availability", "Performance", "Quality", "OEE"]].mean()
oee_by_process = oee_by_process.sort_values("OEE")
print(oee_by_process.round(3))

fig, ax = plt.subplots(figsize=(8, 4.5))
oee_by_process[["Availability", "Performance", "Quality"]].plot(kind="bar", ax=ax)
ax.axhline(0.85, color="green", linestyle="--", linewidth=1, label="World-class OEE benchmark (0.85)")
ax.set_ylabel("Ratio (0-1)")
ax.set_title("OEE components by process")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(f"{REPORTS}/01_oee_components_by_process.png", dpi=150)
plt.show()


### 1.2 OEE trend over time (weekly)

In [ ]:
weekly_oee = production.groupby(["ISOWeek", "Process"])["OEE"].mean().unstack("Process")

fig, ax = plt.subplots(figsize=(10, 4.5))
weekly_oee.plot(ax=ax, marker=".", linewidth=1)
ax.set_ylabel("OEE")
ax.set_xlabel("ISO week")
ax.set_title("Weekly OEE trend by process")
plt.tight_layout()
plt.savefig(f"{REPORTS}/02_oee_trend_weekly.png", dpi=150)
plt.show()


### 1.3 OEE of each machine, within its process

In [ ]:
oee_by_machine = production.groupby(["Process", "MachineId"])["OEE"].mean().reset_index()
oee_by_machine = oee_by_machine.sort_values(["Process", "OEE"])

# one small chart per process, in a 2x2 grid
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for ax, process in zip(axes.flat, production["Process"].unique()):
    process_data = oee_by_machine[oee_by_machine["Process"] == process]
    ax.barh(process_data["MachineId"], process_data["OEE"], color=MAIN_COLOR)
    ax.set_title(process)
    ax.set_xlim(0, 1)
plt.tight_layout()
plt.savefig(f"{REPORTS}/03_oee_by_machine.png", dpi=150)
plt.show()


### 1.4 Downtime Pareto -- planned and unplanned, kept separate

I split this on purpose: if I lump everything into one chart, the
planned events (mold changes, cleaning, meal breaks) always show up on
top, simply because they happen on every order -- and that hides the
stoppage reasons a maintenance team actually needs to fix.


In [ ]:
planned_downtime = downtime[downtime["PlannedStoppage"] == "Yes"]
unplanned_downtime = downtime[downtime["PlannedStoppage"] == "No"]

planned_hours = planned_downtime["DowntimeDurationMin"].sum() / 60
unplanned_hours = unplanned_downtime["DowntimeDurationMin"].sum() / 60

print(f"Planned stoppages:   {len(planned_downtime):,} events, {planned_hours:,.0f} hours")
print(f"Unplanned stoppages: {len(unplanned_downtime):,} events, {unplanned_hours:,.0f} hours")


In [ ]:
def pareto_chart(df, title, file_name, color):
    # sum the minutes lost by reason, from most to least
    minutes_by_reason = df.groupby("StoppageReason")["DowntimeDurationMin"].sum().sort_values(ascending=False)
    cumulative_pct = minutes_by_reason.cumsum() / minutes_by_reason.sum() * 100

    fig, bar_axis = plt.subplots(figsize=(11, 5))
    bar_axis.bar(range(len(minutes_by_reason)), minutes_by_reason.values, color=color)
    bar_axis.set_xticks(range(len(minutes_by_reason)))
    bar_axis.set_xticklabels(minutes_by_reason.index, rotation=75, ha="right", fontsize=8)
    bar_axis.set_ylabel("Minutes lost")

    # a second Y axis, sharing the same X axis, for the cumulative % line
    line_axis = bar_axis.twinx()
    line_axis.plot(range(len(minutes_by_reason)), cumulative_pct.values, color="firebrick", marker="o", markersize=3)
    line_axis.axhline(80, color="gray", linestyle="--", linewidth=1)
    line_axis.set_ylabel("Cumulative %")
    line_axis.set_ylim(0, 105)

    bar_axis.set_title(title)
    plt.tight_layout()
    plt.savefig(f"{REPORTS}/{file_name}", dpi=150)
    plt.show()

    reasons_up_to_80pct = (cumulative_pct <= 80).sum() + 1
    print(f"The top {reasons_up_to_80pct} reasons (of {len(minutes_by_reason)}) "
          f"account for ~80% of the minutes lost in this category.")


pareto_chart(unplanned_downtime, "Pareto of UNPLANNED downtime causes (reliability priorities)",
             "04a_pareto_downtime_unplanned.png", "firebrick")


In [ ]:
pareto_chart(planned_downtime, "Pareto of PLANNED downtime causes (capacity planning context)",
             "04b_pareto_downtime_planned.png", MAIN_COLOR)


## 2. Quality

### 2.1 First Pass Yield (FPY) by product family


In [ ]:
fpy = pd.DataFrame({
    "Cap (Injection Molding)": [(cap_disposition["FinalLotDecision"] == "Approved").mean()],
    "Bottle (Blow Molding)": [(bottle_disposition["FinalLotDecision"] == "Approved").mean()],
    "Ink (Screen Printing)": [(ink_disposition["FinalLotDecision"] == "Approved").mean()],
}).T.rename(columns={0: "FPY"})

print((fpy * 100).round(1))

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(fpy.index, fpy["FPY"], color=["#4C72B0", "#55A868", "#C44E52"])
ax.set_ylim(0, 1)
ax.set_ylabel("First Pass Yield")
ax.set_title("First Pass Yield by product family")
plt.tight_layout()
plt.savefig(f"{REPORTS}/05_fpy_by_product_family.png", dpi=150)
plt.show()


### 2.2 Defect Pareto (attribute inspections)

In [ ]:
cap_defects = cap_attribute.assign(Family="Cap")[["Family", "Characteristic", "DefectsFound"]]
bottle_defects = bottle_attribute.assign(Family="Bottle")[["Family", "Characteristic", "DefectsFound"]]
ink_defects = ink_attribute.assign(Family="Ink")[["Family", "Characteristic", "DefectsFound"]]

all_defects = pd.concat([cap_defects, bottle_defects, ink_defects])
defect_pareto = (
    all_defects.groupby(["Family", "Characteristic"])["DefectsFound"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(10, 5))
labels = [f"{family} - {characteristic}" for family, characteristic in defect_pareto.index]
ax.barh(labels[::-1], defect_pareto.values[::-1], color="indianred")
ax.set_xlabel("Total defects found (sample inspections)")
ax.set_title("Top 15 defect characteristics across all product families")
plt.tight_layout()
plt.savefig(f"{REPORTS}/06_pareto_defects.png", dpi=150)
plt.show()


### 2.3 Process capability (Cpk) heatmap -- caps

In [ ]:
cap_summary = cap_variable.drop_duplicates(["Characteristic", "MachineId", "MoldId"])
cpk_table = cap_summary.pivot_table(index="MachineId", columns="Characteristic", values="Cpk", aggfunc="mean")

fig, ax = plt.subplots(figsize=(7, 4.5))
heatmap = ax.imshow(cpk_table.values, cmap="RdYlGn", vmin=0.8, vmax=2.0, aspect="auto")
ax.set_xticks(range(len(cpk_table.columns)))
ax.set_xticklabels(cpk_table.columns, rotation=45, ha="right")
ax.set_yticks(range(len(cpk_table.index)))
ax.set_yticklabels(cpk_table.index)

# write the Cpk number inside each cell of the heatmap
for row in range(cpk_table.shape[0]):
    for col in range(cpk_table.shape[1]):
        value = cpk_table.values[row, col]
        if not np.isnan(value):
            ax.text(col, row, f"{value:.2f}", ha="center", va="center", fontsize=8)

plt.colorbar(heatmap, ax=ax, label="Cpk")
ax.set_title("Cap process capability (Cpk) -- machine x characteristic")
plt.tight_layout()
plt.savefig(f"{REPORTS}/07_cpk_heatmap_cap.png", dpi=150)
plt.show()


## 3. Maintenance

### 3.1 MTBF and MTTR by machine

Both are computed strictly from `UnplannedFailure` events (genuine
equipment failures), excluding planned changeovers and preventive maintenance.


In [ ]:
run_time_by_machine = production.groupby("MachineId")["RunTimeHours"].sum()
failures = downtime[downtime["UnplannedFailure"]]
failure_count = failures.groupby("MachineId").size()
mttr = failures.groupby("MachineId")["DowntimeDurationMin"].mean() / 60.0

mtbf_mttr = pd.DataFrame({
    "RunTimeHours": run_time_by_machine,
    "FailureCount": failure_count,
    "MTBFHours": run_time_by_machine / failure_count,
    "MTTRHours": mttr,
}).dropna()
mtbf_mttr["TheoreticalAvailability"] = mtbf_mttr["MTBFHours"] / (mtbf_mttr["MTBFHours"] + mtbf_mttr["MTTRHours"])
print(mtbf_mttr.round(2).sort_values("MTBFHours"))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
mtbf_mttr["MTBFHours"].sort_values().plot(kind="barh", ax=axes[0], color="seagreen")
axes[0].set_title("MTBF by machine (hours)")
mtbf_mttr["MTTRHours"].sort_values().plot(kind="barh", ax=axes[1], color="indianred")
axes[1].set_title("MTTR by machine (hours)")
plt.tight_layout()
plt.savefig(f"{REPORTS}/08_mtbf_mttr_by_machine.png", dpi=150)
plt.show()


### 3.2 Planned vs. unplanned downtime split

In [ ]:
split = downtime["PlannedStoppage"].value_counts(normalize=True) * 100

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.pie(split.values, labels=[f"{i}\n{v:.1f}%" for i, v in split.items()],
       colors=["#4C72B0", "#DD8452"], startangle=90)
ax.set_title("Downtime events: planned vs. unplanned")
plt.tight_layout()
plt.savefig(f"{REPORTS}/09_downtime_planned_vs_unplanned.png", dpi=150)
plt.show()

planned_minutes = downtime.loc[downtime["PlannedStoppage"] == "Yes", "DowntimeDurationMin"].sum()
unplanned_minutes = downtime.loc[downtime["PlannedStoppage"] == "No", "DowntimeDurationMin"].sum()
print(f"Minutes lost -- planned: {planned_minutes:,.0f} | unplanned: {unplanned_minutes:,.0f}")


### 3.3 Downtime hours by process

In [ ]:
downtime_by_process = downtime.groupby("Process")["DowntimeDurationMin"].sum().sort_values(ascending=False) / 60

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(downtime_by_process.index, downtime_by_process.values, color="slateblue")
ax.set_ylabel("Total hours lost")
ax.set_title("Total downtime hours by process")
plt.tight_layout()
plt.savefig(f"{REPORTS}/10_downtime_hours_by_process.png", dpi=150)
plt.show()


## Summary of this notebook

- Injection Molding and Blow Molding show lower OEE than Screen
  Printing/Hot Foil Stamping.
- The **unplanned** downtime Pareto (not the mixed one) is what a
  continuous-improvement team should actually look at -- mixing in
  planned events would bury the real problems under routine ones like
  scheduled mold changes.
- The Cpk heatmap shows which machine/characteristic combinations have
  the least capable process (values closer to 0.8, in red).


## 4. Downtime hours -- by every dimension that matters

Everything above already shows downtime in **minutes**, broken down by
reason. This section adds the same breakdown in **hours** (easier to
read at a glance for a shift meeting), across every dimension a
maintenance or production manager would ask about: total, by operator,
by machine, by process, and by stoppage reason.

**A design note on "by operator":** the downtime table itself doesn't
have an `OperatorId` column -- it only has the maintenance technician
who attended the stoppage. To get a meaningful "hours stopped by
operator", I link each stoppage to the work order it interrupted
(`MatchedWorkOrder`, already computed in notebook `02`), and from there
to the operator who was running that order. That's a different
question than "which technician fixed it" -- it answers "which
operator's shift absorbed this downtime", which is what a production
supervisor actually needs when reviewing a shift.


In [ ]:
# link every stoppage to the operator of the work order it interrupted
operator_by_order = production.set_index("WorkOrder")["OperatorId"]
downtime["OperatorId"] = downtime["MatchedWorkOrder"].map(operator_by_order)

total_downtime_hours = downtime["DowntimeDurationMin"].sum() / 60
print(f"Total downtime hours (all reasons, all processes): {total_downtime_hours:,.1f} h")


### 4.1 Downtime hours by operator

In [ ]:
downtime_hours_by_operator = (
    downtime.dropna(subset=["OperatorId"])
    .groupby("OperatorId")["DowntimeDurationMin"].sum() / 60
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
downtime_hours_by_operator.plot(kind="barh", color="firebrick", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Downtime hours (linked to that operator's work orders)")
ax.set_title("Downtime hours by operator")
plt.tight_layout()
plt.savefig(f"{REPORTS}/42_downtime_hours_by_operator.png", dpi=150)
plt.show()


### 4.2 Production hours by operator

In [ ]:
production_hours_by_operator = (
    production.groupby("OperatorId")["RunTimeHours"].sum()
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
production_hours_by_operator.plot(kind="barh", color="seagreen", ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Production (run) hours")
ax.set_title("Production hours by operator")
plt.tight_layout()
plt.savefig(f"{REPORTS}/43_production_hours_by_operator.png", dpi=150)
plt.show()


### 4.3 Downtime hours by machine, by process, and by reason

In [ ]:
downtime_hours_by_machine = (downtime.groupby("MachineId")["DowntimeDurationMin"].sum() / 60).sort_values(ascending=False)
downtime_hours_by_process = (downtime.groupby("Process")["DowntimeDurationMin"].sum() / 60).sort_values(ascending=False)
downtime_hours_by_reason = (downtime.groupby("StoppageReason")["DowntimeDurationMin"].sum() / 60).sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))

downtime_hours_by_machine.plot(kind="barh", ax=axes[0], color="indianred")
axes[0].invert_yaxis()
axes[0].set_title("Downtime hours by machine")
axes[0].set_xlabel("Hours")

downtime_hours_by_process.plot(kind="barh", ax=axes[1], color="slateblue")
axes[1].invert_yaxis()
axes[1].set_title("Downtime hours by process")
axes[1].set_xlabel("Hours")

downtime_hours_by_reason.head(10).plot(kind="barh", ax=axes[2], color="darkorange")
axes[2].invert_yaxis()
axes[2].set_title("Downtime hours by reason (top 10)")
axes[2].set_xlabel("Hours")

plt.tight_layout()
plt.savefig(f"{REPORTS}/44_downtime_hours_by_machine_process_reason.png", dpi=150)
plt.show()

print("Downtime hours by machine:")
print(downtime_hours_by_machine.round(1))
print("\nDowntime hours by process:")
print(downtime_hours_by_process.round(1))


### 4.4 MTBF and MTTR -- highlighted as standalone reliability indicators

Both were already computed earlier in this notebook (section 3.1), from
genuine unplanned failures only. This is the same numbers, pulled out
on their own as the two headline reliability metrics maintenance
engineering cares about most.


In [ ]:
print("MTBF and MTTR by machine, in hours:")
print(mtbf_mttr[["MTBFHours", "MTTRHours"]].round(2).sort_values("MTBFHours"))

print(f"\nPlant-wide average MTBF: {mtbf_mttr['MTBFHours'].mean():.1f} hours")
print(f"Plant-wide average MTTR: {mtbf_mttr['MTTRHours'].mean():.1f} hours")
